In [2]:
from pyspark.sql import functions as F, Window 
from pyspark.sql.types import * 

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 4, Finished, Available, Finished, False)

In [3]:
df_raw = spark.read.format('delta').load('Tables/dbo/bronze_transactions') 
print(f'Bronze rows: {df_raw.count():,}') 
df_raw.printSchema() 

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 5, Finished, Available, Finished, False)

Bronze rows: 6,362,620
root
 |-- step: string (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: string (nullable = true)
 |-- newbalanceOrig: string (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: string (nullable = true)
 |-- newbalanceDest: string (nullable = true)
 |-- isFraud: string (nullable = true)
 |-- isFlaggedFraud: string (nullable = true)



In [4]:
df_renamed = df_raw.select( 
    F.col('step').cast(IntegerType()).alias('step_number'), 
    F.col('type').alias('transaction_type'), 
    F.col('amount').cast(DoubleType()).alias('amount'), 
    F.col('nameOrig').alias('origin_account'), 
    F.col('oldbalanceOrg').cast(DoubleType()).alias('origin_balance_before'), 
    F.col('newbalanceOrig').cast(DoubleType()).alias('origin_balance_after'), 
    F.col('nameDest').alias('destination_account'), 
    F.col('oldbalanceDest').cast(DoubleType()).alias('dest_balance_before'), 
    F.col('newbalanceDest').cast(DoubleType()).alias('dest_balance_after'),
    F.col('isFraud').cast(BooleanType()).alias('is_fraud'),
    F.col('isflaggedFraud').cast(BooleanType()).alias('is_flagged_fraud'))

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 6, Finished, Available, Finished, False)

In [5]:
# ── STEP 3: Calculate balance change metrics ───────────────────── 
df_enriched = df_renamed.withColumn('origin_balance_change',F.col('origin_balance_after') - F.col('origin_balance_before')) \
   .withColumn('balance_discrepancy',F.abs(F.col('origin_balance_before') - F.col('origin_balance_after') - 
F.col('amount'))) \
   .withColumn('transaction_id', 
       F.concat_ws('_', F.col('origin_account'), 
F.col('step_number').cast(StringType()))) 

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 7, Finished, Available, Finished, False)

In [6]:
df_enriched.printSchema()

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 8, Finished, Available, Finished, False)

root
 |-- step_number: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- origin_account: string (nullable = true)
 |-- origin_balance_before: double (nullable = true)
 |-- origin_balance_after: double (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- dest_balance_before: double (nullable = true)
 |-- dest_balance_after: double (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- is_flagged_fraud: boolean (nullable = true)
 |-- origin_balance_change: double (nullable = true)
 |-- balance_discrepancy: double (nullable = true)
 |-- transaction_id: string (nullable = false)



In [7]:
from pyspark.sql import functions as F

base_date = "2025-01-01 00:00:00"

df_enriched = (
    df_enriched
    .withColumn(
        "transaction_datetime",
        F.to_timestamp(
            F.from_unixtime(
                F.unix_timestamp(F.lit(base_date))
                + F.col("step_number") * 3600
            )
        )
    )
    .withColumn(
        "transaction_date",
        F.to_date(F.col("transaction_datetime"))
    )
    .withColumn(
        "transaction_year",
        F.year(F.col("transaction_datetime"))
    )
    .withColumn(
        "transaction_month",
        F.month(F.col("transaction_datetime"))
    )
    .withColumn(
        "transaction_quarter",
        F.quarter(F.col("transaction_datetime"))
    )
    .withColumn(
        "transaction_hour",
        F.hour(F.col("transaction_datetime"))
    )
    .withColumn(
        "day_of_week",
        F.dayofweek(F.col("transaction_datetime"))
    )
    .withColumn(
        "is_weekend",
        F.dayofweek(F.col("transaction_datetime")).isin([1, 7])
    )
)

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 9, Finished, Available, Finished, False)

In [8]:
df_enriched.printSchema()

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 10, Finished, Available, Finished, False)

root
 |-- step_number: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- origin_account: string (nullable = true)
 |-- origin_balance_before: double (nullable = true)
 |-- origin_balance_after: double (nullable = true)
 |-- destination_account: string (nullable = true)
 |-- dest_balance_before: double (nullable = true)
 |-- dest_balance_after: double (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- is_flagged_fraud: boolean (nullable = true)
 |-- origin_balance_change: double (nullable = true)
 |-- balance_discrepancy: double (nullable = true)
 |-- transaction_id: string (nullable = false)
 |-- transaction_datetime: timestamp (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_year: integer (nullable = true)
 |-- transaction_month: integer (nullable = true)
 |-- transaction_quarter: integer (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- day_of_week: in

In [9]:
# ── STEP 4: Data quality flags ────────────────────────────────── 
df_clean = df_enriched \
   .withColumn('dq_negative_amount',    F.col('amount') < 0) \
   .withColumn('dq_zero_amount',        F.col('amount') == 0) \
   .withColumn('dq_null_origin',        F.col('origin_account').isNull()) \
   .withColumn('dq_large_discrepancy',  F.col('balance_discrepancy') > 1) \
   .withColumn('is_valid_transaction', 
       ~F.col('dq_negative_amount') & 
       ~F.col('dq_zero_amount') & 
       ~F.col('dq_null_origin')) 

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 11, Finished, Available, Finished, False)

In [10]:
df_silver = df_clean.filter(F.col('is_valid_transaction') == True) 

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 12, Finished, Available, Finished, False)

In [11]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy(
        "transaction_year",
        "transaction_month"
    ) \
    .saveAsTable("silver_transactions")

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 13, Finished, Available, Finished, False)

In [12]:
 
print(f'Silver transactions written: {df_silver.count():,} rows') 
print(f'Dropped (invalid): {df_raw.count() - df_silver.count():,} rows') 
display(df_silver.limit(5)) 

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 14, Finished, Available, Finished, True)

Silver transactions written: 6,362,604 rows
Dropped (invalid): 16 rows


SynapseWidget(Synapse.DataFrame, 8a26ee66-bbeb-4b4a-a299-ec207c24add7)

StatementMeta(, 968f7115-ddc4-442c-aa31-60b9865a2847, 3, Finished, Available, Finished, False)

DataFrame[]